#### 统计中位数，最大值，均值

In [ ]:
# Human
import pandas as pd
import pyBigWig
import numpy as np
import os

def get_scores(df, bw_path, score_name):
    """
    Extracts conservation scores from a bigWig file.

    Args:
        df (pd.DataFrame): DataFrame containing lncRNA information.
        bw_path (str): Path to the bigWig file.
        score_name (str): Name of the conservation score.

    Returns:
        pd.DataFrame: DataFrame with mean, median, and max scores.
        set: Set of lncRNA_ids without scores.
    """
    scores_df = df.copy()
    scores_df[f'{score_name}_mean_score'] = np.nan
    scores_df[f'{score_name}_median_score'] = np.nan
    scores_df[f'{score_name}_max_score'] = np.nan

    no_score_ids = set()

    try:
        with pyBigWig.open(bw_path) as bw:
            for index, row in df.iterrows():
                chrom = row['chr']
                if chrom in bw.chroms():
                    scores = bw.values(chrom, int(row['start']), int(row['end']), numpy=True)
                    if scores.size > 0 and not np.all(np.isnan(scores)):
                        scores_df.at[index, f'{score_name}_mean_score'] = np.nanmean(scores)
                        scores_df.at[index, f'{score_name}_median_score'] = np.nanmedian(scores)
                        scores_df.at[index, f'{score_name}_max_score'] = np.nanmax(scores)
                    else:
                        no_score_ids.add(row['lncRNA_id'])
                else:
                    no_score_ids.add(row['lncRNA_id'])
    except RuntimeError as e:
        print(f"Failed to open {bw_path}: {e}")

    return scores_df, no_score_ids

def calculate_conservation_features(lncRNAs_csv, conservation_dir):
    """
    Calculates conservation features from all bigWig files in a given directory.

    Args:
        lncRNAs_csv (str): Path to lncRNA CSV file.
        conservation_dir (str): Directory containing bigWig files.

    Outputs:
        - conservation_feature.csv: lncRNAs with conservation scores.
        - no_conservation_lncRNA.csv: lncRNAs without available scores.
    """
    lncRNAs = pd.read_csv(lncRNAs_csv)
    missing_scores_ids = set()

    # List all bigWig files in the directory
    bw_files = [f for f in os.listdir(conservation_dir) if f.endswith(('.bw', '.bigWig'))]

    if not bw_files:
        print("No bigWig files found in the directory.")
        return

    # Initialize lnc_with_score with lncRNA records
    lnc_with_score = lncRNAs.copy()

    # Process each bigWig file
    for bw_file in bw_files:
        bw_path = os.path.join(conservation_dir, bw_file)
        score_name = os.path.splitext(bw_file)[0]  # Use filename as score name

        print(f"Processing {bw_file} ...")

        scores_df, no_score_ids = get_scores(lnc_with_score, bw_path, score_name)
        missing_scores_ids.update(no_score_ids)

        # Merge the new scores into the main DataFrame
        lnc_with_score = pd.merge(
            lnc_with_score,
            scores_df[
                [
                    'lncRNA_id',
                    f'{score_name}_mean_score',
                    f'{score_name}_median_score',
                    f'{score_name}_max_score'
                ]
            ],
            on='lncRNA_id',
            how='left'
        )

    # Remove genomic coordinate columns
    lnc_with_score = lnc_with_score.drop(columns=['chr', 'start', 'end', 'strand'])

    # Remove lncRNAs without any scores
    lnc_with_score = lnc_with_score[~lnc_with_score['lncRNA_id'].isin(missing_scores_ids)]

    # Save outputs
    lnc_with_score.to_csv("human_conservation_feature.csv", index=False)
    pd.Series(list(missing_scores_ids)).to_csv(
        "no_conservation_lncRNA.csv",
        index=False,
        header=["lncRNA_id"]
    )

    print("Conservation scores calculated and saved.")
    print(f"conservation_feature.csv contains {len(lnc_with_score)} records.")
    print(f"no_conservation_lncRNA.csv contains {len(missing_scores_ids)} missing lncRNA IDs.")

# Usage Example
conservation_dir = '../../features/conservation/human'  # Directory containing multiple bigWig files
calculate_conservation_features('../../annotate/human/lncRNA_1-based.csv', conservation_dir)


Processing phyloP.bw ...
Processing phastCons.bw ...
Conservation scores calculated and saved.
conservation_feature.csv contains 35336 records.
no_conservation_lncRNA.csv contains 39 missing lncRNA IDs.


In [9]:
# Mouse
import pandas as pd
import pyBigWig
import numpy as np
import os

def get_scores(df, bw_path, score_name):
    """
    Extracts conservation scores from a bigWig file.

    Args:
        df (pd.DataFrame): DataFrame containing lncRNA information.
        bw_path (str): Path to the bigWig file.
        score_name (str): Name of the conservation score.

    Returns:
        pd.DataFrame: DataFrame with mean, median, and max scores.
        set: Set of lncRNA_ids without scores.
    """
    scores_df = df.copy()
    scores_df[f'{score_name}_mean_score'] = np.nan
    scores_df[f'{score_name}_median_score'] = np.nan
    scores_df[f'{score_name}_max_score'] = np.nan

    no_score_ids = set()

    try:
        with pyBigWig.open(bw_path) as bw:
            for index, row in df.iterrows():
                chrom = row['chr']
                if chrom in bw.chroms():
                    scores = bw.values(chrom, int(row['start']), int(row['end']), numpy=True)
                    if scores.size > 0 and not np.all(np.isnan(scores)):
                        scores_df.at[index, f'{score_name}_mean_score'] = np.nanmean(scores)
                        scores_df.at[index, f'{score_name}_median_score'] = np.nanmedian(scores)
                        scores_df.at[index, f'{score_name}_max_score'] = np.nanmax(scores)
                    else:
                        no_score_ids.add(row['lncRNA_id'])
                else:
                    no_score_ids.add(row['lncRNA_id'])
    except RuntimeError as e:
        print(f"Failed to open {bw_path}: {e}")

    return scores_df, no_score_ids

def calculate_conservation_features(lncRNAs_csv, conservation_dir):
    """
    Calculates conservation features from all bigWig files in a given directory.

    Args:
        lncRNAs_csv (str): Path to lncRNA CSV file.
        conservation_dir (str): Directory containing bigWig files.

    Outputs:
        - conservation_feature.csv: lncRNAs with conservation scores.
        - no_conservation_lncRNA.csv: lncRNAs without available scores.
    """
    lncRNAs = pd.read_csv(lncRNAs_csv)
    missing_scores_ids = set()

    # List all bigWig files in the directory
    bw_files = [f for f in os.listdir(conservation_dir) if f.endswith(('.bw', '.bigWig'))]

    if not bw_files:
        print("No bigWig files found in the directory.")
        return

    # Initialize lnc_with_score with lncRNA records
    lnc_with_score = lncRNAs.copy()

    # Process each bigWig file
    for bw_file in bw_files:
        bw_path = os.path.join(conservation_dir, bw_file)
        score_name = os.path.splitext(bw_file)[0]  # Use filename as score name

        print(f"Processing {bw_file} ...")

        scores_df, no_score_ids = get_scores(lnc_with_score, bw_path, score_name)
        missing_scores_ids.update(no_score_ids)

        # Merge the new scores into the main DataFrame
        lnc_with_score = pd.merge(
            lnc_with_score,
            scores_df[
                [
                    'lncRNA_id',
                    f'{score_name}_mean_score',
                    f'{score_name}_median_score',
                    f'{score_name}_max_score'
                ]
            ],
            on='lncRNA_id',
            how='left'
        )

    # Remove genomic coordinate columns
    lnc_with_score = lnc_with_score.drop(columns=['chr', 'start', 'end', 'strand'])

    # Remove lncRNAs without any scores
    lnc_with_score = lnc_with_score[~lnc_with_score['lncRNA_id'].isin(missing_scores_ids)]

    # Save outputs
    lnc_with_score.to_csv("mouse_conservation_feature.csv", index=False)
    pd.Series(list(missing_scores_ids)).to_csv(
        "no_conservation_lncRNA.csv",
        index=False,
        header=["lncRNA_id"]
    )

    print("Conservation scores calculated and saved.")
    print(f"conservation_feature.csv contains {len(lnc_with_score)} records.")
    print(f"no_conservation_lncRNA.csv contains {len(missing_scores_ids)} missing lncRNA IDs.")

# Usage Example
conservation_dir = '../../features/conservation/mouse'  # Directory containing multiple bigWig files
calculate_conservation_features('../../annotate/mouse/lncRNA_1-based.csv', conservation_dir)


Processing phyloP.bw ...
Processing phastCons.bw ...
Conservation scores calculated and saved.
conservation_feature.csv contains 28815 records.
no_conservation_lncRNA.csv contains 210 missing lncRNA IDs.


In [7]:
# Human
import pandas as pd
from scipy.stats import mannwhitneyu

def compare_conservation_features(feature_csv, ess_file_dict, output_csv):
    """
    Compare conservation features between essential lncRNAs and background lncRNAs.

    Args:
        feature_csv (str): Path to conservation_feature.csv.
        ess_file_dict (dict): Dictionary of tissue -> essential lncRNA file path.
        output_csv (str): Output CSV file path.

    Output:
        A CSV file containing summary statistics and Mann-Whitney U test results.
    """
    # Load conservation feature table
    df = pd.read_csv(feature_csv)

    # Assume the first column is lncRNA_id
    if 'lncRNA_id' not in df.columns:
        raise ValueError("Input feature table must contain a column named 'lncRNA_id'.")

    all_lnc_set = set(df['lncRNA_id'])

    # Identify all score columns
    mean_cols = [col for col in df.columns if col.endswith('_mean_score')]
    median_cols = [col for col in df.columns if col.endswith('_median_score')]
    max_cols = [col for col in df.columns if col.endswith('_max_score')]

    score_cols = mean_cols + median_cols + max_cols

    results = []

    for tissue, ess_path in ess_file_dict.items():
        print(f'Processing tissue: {tissue}')

        # Load essential lncRNAs
        ess_df = pd.read_csv(ess_path, header=None)
        ess_set = set(ess_df.iloc[:, 0])

        # Background = all non-essential lncRNAs
        bg_set = all_lnc_set - ess_set

        # Keep only lncRNAs present in the feature table
        ess_valid = ess_set & all_lnc_set
        bg_valid = bg_set & all_lnc_set

        ess_data = df[df['lncRNA_id'].isin(ess_valid)].copy()
        bg_data = df[df['lncRNA_id'].isin(bg_valid)].copy()

        for col in score_cols:
            ess_values = ess_data[col].dropna()
            bg_values = bg_data[col].dropna()

            if len(ess_values) == 0 or len(bg_values) == 0:
                print(f"Skipping {tissue} - {col}: no valid values.")
                continue

            # One-sided Mann-Whitney U test: essential > background
            u_stat, p_value = mannwhitneyu(
                ess_values,
                bg_values,
                alternative='greater'
            )

            results.append({
                'tissue': tissue,
                'feature': col,
                'n_ess': len(ess_values),
                'n_bg': len(bg_values),
                'mean_ess': ess_values.mean(),
                'median_ess': ess_values.median(),
                'mean_bg': bg_values.mean(),
                'median_bg': bg_values.median(),
                'u_stat': u_stat,
                'p_value': p_value
            })

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    print(f'Results saved to {output_csv}')


# =========================
# Example usage
# =========================

ess_file_dict = {
    'heart': '../ess_number/filtered/human/BC_top40pct_human_heart_esslnc.csv',
    'lung': '../ess_number/filtered/human/BC_top40pct_human_lung_esslnc.csv',
    'stomach': '../ess_number/filtered/human/BC_top40pct_human_stomach_esslnc.csv'
}

compare_conservation_features(
    feature_csv='human_conservation_feature.csv',
    ess_file_dict=ess_file_dict,
    output_csv='human_conservation_comparison.csv'
)


Processing tissue: heart
Processing tissue: lung
Processing tissue: stomach
Results saved to human_conservation_comparison.csv


In [10]:
# Mouse
import pandas as pd
from scipy.stats import mannwhitneyu

def compare_conservation_features(feature_csv, ess_file_dict, output_csv):
    """
    Compare conservation features between essential lncRNAs and background lncRNAs.

    Args:
        feature_csv (str): Path to conservation_feature.csv.
        ess_file_dict (dict): Dictionary of tissue -> essential lncRNA file path.
        output_csv (str): Output CSV file path.

    Output:
        A CSV file containing summary statistics and Mann-Whitney U test results.
    """
    # Load conservation feature table
    df = pd.read_csv(feature_csv)

    # Assume the first column is lncRNA_id
    if 'lncRNA_id' not in df.columns:
        raise ValueError("Input feature table must contain a column named 'lncRNA_id'.")

    all_lnc_set = set(df['lncRNA_id'])

    # Identify all score columns
    mean_cols = [col for col in df.columns if col.endswith('_mean_score')]
    median_cols = [col for col in df.columns if col.endswith('_median_score')]
    max_cols = [col for col in df.columns if col.endswith('_max_score')]

    score_cols = mean_cols + median_cols + max_cols

    results = []

    for tissue, ess_path in ess_file_dict.items():
        print(f'Processing tissue: {tissue}')

        # Load essential lncRNAs
        ess_df = pd.read_csv(ess_path, header=None)
        ess_set = set(ess_df.iloc[:, 0])

        # Background = all non-essential lncRNAs
        bg_set = all_lnc_set - ess_set

        # Keep only lncRNAs present in the feature table
        ess_valid = ess_set & all_lnc_set
        bg_valid = bg_set & all_lnc_set

        ess_data = df[df['lncRNA_id'].isin(ess_valid)].copy()
        bg_data = df[df['lncRNA_id'].isin(bg_valid)].copy()

        for col in score_cols:
            ess_values = ess_data[col].dropna()
            bg_values = bg_data[col].dropna()

            if len(ess_values) == 0 or len(bg_values) == 0:
                print(f"Skipping {tissue} - {col}: no valid values.")
                continue

            # One-sided Mann-Whitney U test: essential > background
            u_stat, p_value = mannwhitneyu(
                ess_values,
                bg_values,
                alternative='greater'
            )

            results.append({
                'tissue': tissue,
                'feature': col,
                'n_ess': len(ess_values),
                'n_bg': len(bg_values),
                'mean_ess': ess_values.mean(),
                'median_ess': ess_values.median(),
                'mean_bg': bg_values.mean(),
                'median_bg': bg_values.median(),
                'u_stat': u_stat,
                'p_value': p_value
            })

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    print(f'Results saved to {output_csv}')


# =========================
# Example usage
# =========================

ess_file_dict = {
    'heart': '../ess_number/filtered/mouse/BC_top60pct_mouse_heart_esslnc.csv',
    'lung': '../ess_number/filtered/mouse/BC_top60pct_mouse_lung_esslnc.csv',
    'brain': '../ess_number/filtered/mouse/BC_top60pct_mouse_brain_esslnc.csv'
}

compare_conservation_features(
    feature_csv='mouse_conservation_feature.csv',
    ess_file_dict=ess_file_dict,
    output_csv='mouse_conservation_comparison.csv'
)


Processing tissue: heart
Processing tissue: lung
Processing tissue: brain
Results saved to mouse_conservation_comparison.csv


找同源

In [14]:
# human left ventricle -- mouse heart
import pandas as pd
import re

homo = pd.read_csv("homo.csv")

h_esl = pd.read_csv("../ess_number/filtered/human/BC_top40pct_human_heart_esslnc.csv", header=None)
m_esl = pd.read_csv("../ess_number/filtered/mouse/BC_top60pct_mouse_heart_esslnc.csv", header=None)

h_mapping = pd.read_csv("../../data/LPI/human/lncRNA_mapping.csv")
h_lnc = pd.read_csv("../../data/LPI/human/lncRNA.csv")
m_mapping = pd.read_csv("../../data/LPI/mouse/lncRNA_mapping.csv")
m_lnc = pd.read_csv("../../data/LPI/mouse/lncRNA.csv")


def extract_ensembl_id(x, species="human"):
    """
    Extract Ensembl gene stable ID from gene_id string.
    Human: ENSG...
    Mouse: ENSMUSG...
    """
    if pd.isna(x):
        return None

    x = str(x)

    if species == "human":
        m = re.search(r"(ENSG\d+)", x)
    else:
        m = re.search(r"(ENSMUSG\d+)", x)

    return m.group(1) if m else None


def build_member_annotation(esl_df, mapping_df, lnc_df, species="human"):
    """
    Expand merged lncRNA IDs to member-level annotations.
    """
    esl_df = esl_df.copy()
    esl_df.columns = ["lncRNA_id"]

    # Keep only essential merged IDs
    mapping_sub = mapping_df[mapping_df["lncRNA_id"].isin(esl_df["lncRNA_id"])].copy()

    # Merge with original lncRNA table using member_id -> identifier
    member_df = mapping_sub.merge(
        lnc_df[["identifier", "gene_name", "gene_id"]],
        left_on="member_id",
        right_on="identifier",
        how="left"
    )

    # Extract Ensembl gene ID
    member_df["ensembl_gene_id"] = member_df["gene_id"].apply(
        lambda x: extract_ensembl_id(x, species=species)
    )

    # Clean gene name
    member_df["gene_name"] = member_df["gene_name"].astype(str).str.strip()
    member_df.loc[member_df["gene_name"].isin(["", "nan", "None"]), "gene_name"] = pd.NA

    return member_df


def make_homology_key_table(homo_df):
    """
    Build a unified homology key table with both Ensembl-based and gene-name-based keys.
    """
    homo_sub = homo_df[
        ["Gene stable ID", "Gene name", "Mouse gene stable ID", "Mouse gene name"]
    ].copy()

    for col in ["Gene stable ID", "Gene name", "Mouse gene stable ID", "Mouse gene name"]:
        homo_sub[col] = homo_sub[col].astype(str).str.strip()
        homo_sub.loc[homo_sub[col].isin(["", "nan", "None"]), col] = pd.NA

    return homo_sub


def match_by_ensembl(h_member_df, m_member_df, homo_df):
    """
    Match human and mouse members through homology table using Ensembl IDs.
    """
    h_valid = h_member_df.dropna(subset=["ensembl_gene_id"]).copy()
    m_valid = m_member_df.dropna(subset=["ensembl_gene_id"]).copy()

    # Human member -> homo
    h_homo = h_valid.merge(
        homo_df,
        left_on="ensembl_gene_id",
        right_on="Gene stable ID",
        how="inner"
    )

    # Mouse member -> homo
    m_homo = m_valid.merge(
        homo_df,
        left_on="ensembl_gene_id",
        right_on="Mouse gene stable ID",
        how="inner"
    )

    # Join on the same homology pair
    pairs = h_homo.merge(
        m_homo,
        on=["Gene stable ID", "Mouse gene stable ID", "Gene name", "Mouse gene name"],
        suffixes=("_human", "_mouse"),
        how="inner"
    )

    pairs["match_basis"] = "ensembl"

    return pairs


def match_by_gene_name(h_member_df, m_member_df, homo_df):
    """
    Match human and mouse members through homology table using gene names.
    """
    h_valid = h_member_df.dropna(subset=["gene_name"]).copy()
    m_valid = m_member_df.dropna(subset=["gene_name"]).copy()

    # Human member -> homo
    h_homo = h_valid.merge(
        homo_df,
        left_on="gene_name",
        right_on="Gene name",
        how="inner"
    )

    # Mouse member -> homo
    m_homo = m_valid.merge(
        homo_df,
        left_on="gene_name",
        right_on="Mouse gene name",
        how="inner"
    )

    # Join on the same homology pair
    pairs = h_homo.merge(
        m_homo,
        on=["Gene stable ID", "Mouse gene stable ID", "Gene name", "Mouse gene name"],
        suffixes=("_human", "_mouse"),
        how="inner"
    )

    pairs["match_basis"] = "gene_name"

    return pairs


def format_member_pairs(pairs_df):
    """
    Keep and rename useful columns.
    """
    out = pairs_df[
        [
            "lncRNA_id_human",
            "member_id_human",
            "identifier_human",
            "gene_name_human",
            "gene_id_human",
            "ensembl_gene_id_human",

            "Gene stable ID",
            "Gene name",
            "Mouse gene stable ID",
            "Mouse gene name",

            "lncRNA_id_mouse",
            "member_id_mouse",
            "identifier_mouse",
            "gene_name_mouse",
            "gene_id_mouse",
            "ensembl_gene_id_mouse",

            "match_basis"
        ]
    ].copy()

    out = out.rename(columns={
        "lncRNA_id_human": "human_lncRNA_id",
        "member_id_human": "human_member_id",
        "identifier_human": "human_identifier",
        "gene_name_human": "human_gene_name_in_lnc",
        "gene_id_human": "human_gene_id_raw",
        "ensembl_gene_id_human": "human_ensembl_gene_id",

        "Gene stable ID": "homo_human_gene_stable_id",
        "Gene name": "homo_human_gene_name",
        "Mouse gene stable ID": "homo_mouse_gene_stable_id",
        "Mouse gene name": "homo_mouse_gene_name",

        "lncRNA_id_mouse": "mouse_lncRNA_id",
        "member_id_mouse": "mouse_member_id",
        "identifier_mouse": "mouse_identifier",
        "gene_name_mouse": "mouse_gene_name_in_lnc",
        "gene_id_mouse": "mouse_gene_id_raw",
        "ensembl_gene_id_mouse": "mouse_ensembl_gene_id"
    })

    return out.drop_duplicates()


# =========================
# 1. Build member-level annotation tables
# =========================

h_member = build_member_annotation(h_esl, h_mapping, h_lnc, species="human")
m_member = build_member_annotation(m_esl, m_mapping, m_lnc, species="mouse")

# =========================
# 2. Build homology key table
# =========================

homo_key = make_homology_key_table(homo)

# =========================
# 3. Match by Ensembl ID and by gene name
# =========================

pairs_ensembl = match_by_ensembl(h_member, m_member, homo_key)
pairs_gene_name = match_by_gene_name(h_member, m_member, homo_key)

# Combine both matching results
member_pairs = pd.concat([pairs_ensembl, pairs_gene_name], ignore_index=True)
member_pairs = format_member_pairs(member_pairs)

# Remove exact duplicate member-pair records
member_pairs = member_pairs.drop_duplicates()

# =========================
# 4. Collapse to merged-level homologous lncRNA pairs
#    Rule:
#    if one human member and one mouse member are homologous,
#    then their merged lncRNA_ids are considered homologous
# =========================

merged_pairs = (
    member_pairs.groupby(["human_lncRNA_id", "mouse_lncRNA_id"])
    .agg(
        n_homologous_member_pairs=("match_basis", "count"),
        human_member_count=("human_member_id", pd.Series.nunique),
        mouse_member_count=("mouse_member_id", pd.Series.nunique),
        match_basis=("match_basis", lambda x: ";".join(sorted(set(x)))),
        human_ensembl_ids=("human_ensembl_gene_id", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        mouse_ensembl_ids=("mouse_ensembl_gene_id", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        human_gene_names=("human_gene_name_in_lnc", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        mouse_gene_names=("mouse_gene_name_in_lnc", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        homo_human_gene_names=("homo_human_gene_name", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        homo_mouse_gene_names=("homo_mouse_gene_name", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)]))))
    )
    .reset_index()
)

# =========================
# 5. Save outputs
# =========================

member_pairs.to_csv("heart_member_level_homolog_pairs.csv", index=False)
merged_pairs.to_csv("heart_merged_level_homolog_pairs.csv", index=False)

print("Done.")
print(f"Human essential merged lncRNAs: {h_esl.shape[0]}")
print(f"Mouse essential merged lncRNAs: {m_esl.shape[0]}")
print(f"Member-level homologous pairs: {member_pairs.shape[0]}")
print(f"Merged-level homologous lncRNA pairs: {merged_pairs.shape[0]}")
print(f"Ensembl-based matches: {(member_pairs['match_basis'] == 'ensembl').sum()}")
print(f"Gene-name-based matches: {(member_pairs['match_basis'] == 'gene_name').sum()}")




Done.
Human essential merged lncRNAs: 6727
Mouse essential merged lncRNAs: 4154
Member-level homologous pairs: 316
Merged-level homologous lncRNA pairs: 235
Ensembl-based matches: 0
Gene-name-based matches: 316


In [15]:
# human left lung -- mouse lung
import pandas as pd
import re

homo = pd.read_csv("homo.csv")

h_esl = pd.read_csv("../ess_number/filtered/human/BC_top40pct_human_lung_esslnc.csv", header=None)
m_esl = pd.read_csv("../ess_number/filtered/mouse/BC_top60pct_mouse_lung_esslnc.csv", header=None)

h_mapping = pd.read_csv("../../data/LPI/human/lncRNA_mapping.csv")
h_lnc = pd.read_csv("../../data/LPI/human/lncRNA.csv")
m_mapping = pd.read_csv("../../data/LPI/mouse/lncRNA_mapping.csv")
m_lnc = pd.read_csv("../../data/LPI/mouse/lncRNA.csv")


def extract_ensembl_id(x, species="human"):
    """
    Extract Ensembl gene stable ID from gene_id string.
    Human: ENSG...
    Mouse: ENSMUSG...
    """
    if pd.isna(x):
        return None

    x = str(x)

    if species == "human":
        m = re.search(r"(ENSG\d+)", x)
    else:
        m = re.search(r"(ENSMUSG\d+)", x)

    return m.group(1) if m else None


def build_member_annotation(esl_df, mapping_df, lnc_df, species="human"):
    """
    Expand merged lncRNA IDs to member-level annotations.
    """
    esl_df = esl_df.copy()
    esl_df.columns = ["lncRNA_id"]

    # Keep only essential merged IDs
    mapping_sub = mapping_df[mapping_df["lncRNA_id"].isin(esl_df["lncRNA_id"])].copy()

    # Merge with original lncRNA table using member_id -> identifier
    member_df = mapping_sub.merge(
        lnc_df[["identifier", "gene_name", "gene_id"]],
        left_on="member_id",
        right_on="identifier",
        how="left"
    )

    # Extract Ensembl gene ID
    member_df["ensembl_gene_id"] = member_df["gene_id"].apply(
        lambda x: extract_ensembl_id(x, species=species)
    )

    # Clean gene name
    member_df["gene_name"] = member_df["gene_name"].astype(str).str.strip()
    member_df.loc[member_df["gene_name"].isin(["", "nan", "None"]), "gene_name"] = pd.NA

    return member_df


def make_homology_key_table(homo_df):
    """
    Build a unified homology key table with both Ensembl-based and gene-name-based keys.
    """
    homo_sub = homo_df[
        ["Gene stable ID", "Gene name", "Mouse gene stable ID", "Mouse gene name"]
    ].copy()

    for col in ["Gene stable ID", "Gene name", "Mouse gene stable ID", "Mouse gene name"]:
        homo_sub[col] = homo_sub[col].astype(str).str.strip()
        homo_sub.loc[homo_sub[col].isin(["", "nan", "None"]), col] = pd.NA

    return homo_sub


def match_by_ensembl(h_member_df, m_member_df, homo_df):
    """
    Match human and mouse members through homology table using Ensembl IDs.
    """
    h_valid = h_member_df.dropna(subset=["ensembl_gene_id"]).copy()
    m_valid = m_member_df.dropna(subset=["ensembl_gene_id"]).copy()

    # Human member -> homo
    h_homo = h_valid.merge(
        homo_df,
        left_on="ensembl_gene_id",
        right_on="Gene stable ID",
        how="inner"
    )

    # Mouse member -> homo
    m_homo = m_valid.merge(
        homo_df,
        left_on="ensembl_gene_id",
        right_on="Mouse gene stable ID",
        how="inner"
    )

    # Join on the same homology pair
    pairs = h_homo.merge(
        m_homo,
        on=["Gene stable ID", "Mouse gene stable ID", "Gene name", "Mouse gene name"],
        suffixes=("_human", "_mouse"),
        how="inner"
    )

    pairs["match_basis"] = "ensembl"

    return pairs


def match_by_gene_name(h_member_df, m_member_df, homo_df):
    """
    Match human and mouse members through homology table using gene names.
    """
    h_valid = h_member_df.dropna(subset=["gene_name"]).copy()
    m_valid = m_member_df.dropna(subset=["gene_name"]).copy()

    # Human member -> homo
    h_homo = h_valid.merge(
        homo_df,
        left_on="gene_name",
        right_on="Gene name",
        how="inner"
    )

    # Mouse member -> homo
    m_homo = m_valid.merge(
        homo_df,
        left_on="gene_name",
        right_on="Mouse gene name",
        how="inner"
    )

    # Join on the same homology pair
    pairs = h_homo.merge(
        m_homo,
        on=["Gene stable ID", "Mouse gene stable ID", "Gene name", "Mouse gene name"],
        suffixes=("_human", "_mouse"),
        how="inner"
    )

    pairs["match_basis"] = "gene_name"

    return pairs


def format_member_pairs(pairs_df):
    """
    Keep and rename useful columns.
    """
    out = pairs_df[
        [
            "lncRNA_id_human",
            "member_id_human",
            "identifier_human",
            "gene_name_human",
            "gene_id_human",
            "ensembl_gene_id_human",

            "Gene stable ID",
            "Gene name",
            "Mouse gene stable ID",
            "Mouse gene name",

            "lncRNA_id_mouse",
            "member_id_mouse",
            "identifier_mouse",
            "gene_name_mouse",
            "gene_id_mouse",
            "ensembl_gene_id_mouse",

            "match_basis"
        ]
    ].copy()

    out = out.rename(columns={
        "lncRNA_id_human": "human_lncRNA_id",
        "member_id_human": "human_member_id",
        "identifier_human": "human_identifier",
        "gene_name_human": "human_gene_name_in_lnc",
        "gene_id_human": "human_gene_id_raw",
        "ensembl_gene_id_human": "human_ensembl_gene_id",

        "Gene stable ID": "homo_human_gene_stable_id",
        "Gene name": "homo_human_gene_name",
        "Mouse gene stable ID": "homo_mouse_gene_stable_id",
        "Mouse gene name": "homo_mouse_gene_name",

        "lncRNA_id_mouse": "mouse_lncRNA_id",
        "member_id_mouse": "mouse_member_id",
        "identifier_mouse": "mouse_identifier",
        "gene_name_mouse": "mouse_gene_name_in_lnc",
        "gene_id_mouse": "mouse_gene_id_raw",
        "ensembl_gene_id_mouse": "mouse_ensembl_gene_id"
    })

    return out.drop_duplicates()


# =========================
# 1. Build member-level annotation tables
# =========================

h_member = build_member_annotation(h_esl, h_mapping, h_lnc, species="human")
m_member = build_member_annotation(m_esl, m_mapping, m_lnc, species="mouse")

# =========================
# 2. Build homology key table
# =========================

homo_key = make_homology_key_table(homo)

# =========================
# 3. Match by Ensembl ID and by gene name
# =========================

pairs_ensembl = match_by_ensembl(h_member, m_member, homo_key)
pairs_gene_name = match_by_gene_name(h_member, m_member, homo_key)

# Combine both matching results
member_pairs = pd.concat([pairs_ensembl, pairs_gene_name], ignore_index=True)
member_pairs = format_member_pairs(member_pairs)

# Remove exact duplicate member-pair records
member_pairs = member_pairs.drop_duplicates()

# =========================
# 4. Collapse to merged-level homologous lncRNA pairs
#    Rule:
#    if one human member and one mouse member are homologous,
#    then their merged lncRNA_ids are considered homologous
# =========================

merged_pairs = (
    member_pairs.groupby(["human_lncRNA_id", "mouse_lncRNA_id"])
    .agg(
        n_homologous_member_pairs=("match_basis", "count"),
        human_member_count=("human_member_id", pd.Series.nunique),
        mouse_member_count=("mouse_member_id", pd.Series.nunique),
        match_basis=("match_basis", lambda x: ";".join(sorted(set(x)))),
        human_ensembl_ids=("human_ensembl_gene_id", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        mouse_ensembl_ids=("mouse_ensembl_gene_id", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        human_gene_names=("human_gene_name_in_lnc", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        mouse_gene_names=("mouse_gene_name_in_lnc", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        homo_human_gene_names=("homo_human_gene_name", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        homo_mouse_gene_names=("homo_mouse_gene_name", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)]))))
    )
    .reset_index()
)

# =========================
# 5. Save outputs
# =========================

member_pairs.to_csv("heart_member_level_homolog_pairs.csv", index=False)
merged_pairs.to_csv("heart_merged_level_homolog_pairs.csv", index=False)

print("Done.")
print(f"Human essential merged lncRNAs: {h_esl.shape[0]}")
print(f"Mouse essential merged lncRNAs: {m_esl.shape[0]}")
print(f"Member-level homologous pairs: {member_pairs.shape[0]}")
print(f"Merged-level homologous lncRNA pairs: {merged_pairs.shape[0]}")
print(f"Ensembl-based matches: {(member_pairs['match_basis'] == 'ensembl').sum()}")
print(f"Gene-name-based matches: {(member_pairs['match_basis'] == 'gene_name').sum()}")




Done.
Human essential merged lncRNAs: 6471
Mouse essential merged lncRNAs: 6202
Member-level homologous pairs: 473
Merged-level homologous lncRNA pairs: 344
Ensembl-based matches: 1
Gene-name-based matches: 472


In [16]:
# human common -- mouse common
import pandas as pd
import re

homo = pd.read_csv("homo.csv")

h_esl = pd.read_csv("../ess_number/filtered/human/BC_top40pct_human_common_esslnc.csv", header=None)
m_esl = pd.read_csv("../ess_number/filtered/mouse/BC_top60pct_mouse_common_esslnc.csv", header=None)

h_mapping = pd.read_csv("../../data/LPI/human/lncRNA_mapping.csv")
h_lnc = pd.read_csv("../../data/LPI/human/lncRNA.csv")
m_mapping = pd.read_csv("../../data/LPI/mouse/lncRNA_mapping.csv")
m_lnc = pd.read_csv("../../data/LPI/mouse/lncRNA.csv")


def extract_ensembl_id(x, species="human"):
    """
    Extract Ensembl gene stable ID from gene_id string.
    Human: ENSG...
    Mouse: ENSMUSG...
    """
    if pd.isna(x):
        return None

    x = str(x)

    if species == "human":
        m = re.search(r"(ENSG\d+)", x)
    else:
        m = re.search(r"(ENSMUSG\d+)", x)

    return m.group(1) if m else None


def build_member_annotation(esl_df, mapping_df, lnc_df, species="human"):
    """
    Expand merged lncRNA IDs to member-level annotations.
    """
    esl_df = esl_df.copy()
    esl_df.columns = ["lncRNA_id"]

    # Keep only essential merged IDs
    mapping_sub = mapping_df[mapping_df["lncRNA_id"].isin(esl_df["lncRNA_id"])].copy()

    # Merge with original lncRNA table using member_id -> identifier
    member_df = mapping_sub.merge(
        lnc_df[["identifier", "gene_name", "gene_id"]],
        left_on="member_id",
        right_on="identifier",
        how="left"
    )

    # Extract Ensembl gene ID
    member_df["ensembl_gene_id"] = member_df["gene_id"].apply(
        lambda x: extract_ensembl_id(x, species=species)
    )

    # Clean gene name
    member_df["gene_name"] = member_df["gene_name"].astype(str).str.strip()
    member_df.loc[member_df["gene_name"].isin(["", "nan", "None"]), "gene_name"] = pd.NA

    return member_df


def make_homology_key_table(homo_df):
    """
    Build a unified homology key table with both Ensembl-based and gene-name-based keys.
    """
    homo_sub = homo_df[
        ["Gene stable ID", "Gene name", "Mouse gene stable ID", "Mouse gene name"]
    ].copy()

    for col in ["Gene stable ID", "Gene name", "Mouse gene stable ID", "Mouse gene name"]:
        homo_sub[col] = homo_sub[col].astype(str).str.strip()
        homo_sub.loc[homo_sub[col].isin(["", "nan", "None"]), col] = pd.NA

    return homo_sub


def match_by_ensembl(h_member_df, m_member_df, homo_df):
    """
    Match human and mouse members through homology table using Ensembl IDs.
    """
    h_valid = h_member_df.dropna(subset=["ensembl_gene_id"]).copy()
    m_valid = m_member_df.dropna(subset=["ensembl_gene_id"]).copy()

    # Human member -> homo
    h_homo = h_valid.merge(
        homo_df,
        left_on="ensembl_gene_id",
        right_on="Gene stable ID",
        how="inner"
    )

    # Mouse member -> homo
    m_homo = m_valid.merge(
        homo_df,
        left_on="ensembl_gene_id",
        right_on="Mouse gene stable ID",
        how="inner"
    )

    # Join on the same homology pair
    pairs = h_homo.merge(
        m_homo,
        on=["Gene stable ID", "Mouse gene stable ID", "Gene name", "Mouse gene name"],
        suffixes=("_human", "_mouse"),
        how="inner"
    )

    pairs["match_basis"] = "ensembl"

    return pairs


def match_by_gene_name(h_member_df, m_member_df, homo_df):
    """
    Match human and mouse members through homology table using gene names.
    """
    h_valid = h_member_df.dropna(subset=["gene_name"]).copy()
    m_valid = m_member_df.dropna(subset=["gene_name"]).copy()

    # Human member -> homo
    h_homo = h_valid.merge(
        homo_df,
        left_on="gene_name",
        right_on="Gene name",
        how="inner"
    )

    # Mouse member -> homo
    m_homo = m_valid.merge(
        homo_df,
        left_on="gene_name",
        right_on="Mouse gene name",
        how="inner"
    )

    # Join on the same homology pair
    pairs = h_homo.merge(
        m_homo,
        on=["Gene stable ID", "Mouse gene stable ID", "Gene name", "Mouse gene name"],
        suffixes=("_human", "_mouse"),
        how="inner"
    )

    pairs["match_basis"] = "gene_name"

    return pairs


def format_member_pairs(pairs_df):
    """
    Keep and rename useful columns.
    """
    out = pairs_df[
        [
            "lncRNA_id_human",
            "member_id_human",
            "identifier_human",
            "gene_name_human",
            "gene_id_human",
            "ensembl_gene_id_human",

            "Gene stable ID",
            "Gene name",
            "Mouse gene stable ID",
            "Mouse gene name",

            "lncRNA_id_mouse",
            "member_id_mouse",
            "identifier_mouse",
            "gene_name_mouse",
            "gene_id_mouse",
            "ensembl_gene_id_mouse",

            "match_basis"
        ]
    ].copy()

    out = out.rename(columns={
        "lncRNA_id_human": "human_lncRNA_id",
        "member_id_human": "human_member_id",
        "identifier_human": "human_identifier",
        "gene_name_human": "human_gene_name_in_lnc",
        "gene_id_human": "human_gene_id_raw",
        "ensembl_gene_id_human": "human_ensembl_gene_id",

        "Gene stable ID": "homo_human_gene_stable_id",
        "Gene name": "homo_human_gene_name",
        "Mouse gene stable ID": "homo_mouse_gene_stable_id",
        "Mouse gene name": "homo_mouse_gene_name",

        "lncRNA_id_mouse": "mouse_lncRNA_id",
        "member_id_mouse": "mouse_member_id",
        "identifier_mouse": "mouse_identifier",
        "gene_name_mouse": "mouse_gene_name_in_lnc",
        "gene_id_mouse": "mouse_gene_id_raw",
        "ensembl_gene_id_mouse": "mouse_ensembl_gene_id"
    })

    return out.drop_duplicates()


# =========================
# 1. Build member-level annotation tables
# =========================

h_member = build_member_annotation(h_esl, h_mapping, h_lnc, species="human")
m_member = build_member_annotation(m_esl, m_mapping, m_lnc, species="mouse")

# =========================
# 2. Build homology key table
# =========================

homo_key = make_homology_key_table(homo)

# =========================
# 3. Match by Ensembl ID and by gene name
# =========================

pairs_ensembl = match_by_ensembl(h_member, m_member, homo_key)
pairs_gene_name = match_by_gene_name(h_member, m_member, homo_key)

# Combine both matching results
member_pairs = pd.concat([pairs_ensembl, pairs_gene_name], ignore_index=True)
member_pairs = format_member_pairs(member_pairs)

# Remove exact duplicate member-pair records
member_pairs = member_pairs.drop_duplicates()

# =========================
# 4. Collapse to merged-level homologous lncRNA pairs
#    Rule:
#    if one human member and one mouse member are homologous,
#    then their merged lncRNA_ids are considered homologous
# =========================

merged_pairs = (
    member_pairs.groupby(["human_lncRNA_id", "mouse_lncRNA_id"])
    .agg(
        n_homologous_member_pairs=("match_basis", "count"),
        human_member_count=("human_member_id", pd.Series.nunique),
        mouse_member_count=("mouse_member_id", pd.Series.nunique),
        match_basis=("match_basis", lambda x: ";".join(sorted(set(x)))),
        human_ensembl_ids=("human_ensembl_gene_id", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        mouse_ensembl_ids=("mouse_ensembl_gene_id", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        human_gene_names=("human_gene_name_in_lnc", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        mouse_gene_names=("mouse_gene_name_in_lnc", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        homo_human_gene_names=("homo_human_gene_name", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
        homo_mouse_gene_names=("homo_mouse_gene_name", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)]))))
    )
    .reset_index()
)

# =========================
# 5. Save outputs
# =========================

member_pairs.to_csv("heart_member_level_homolog_pairs.csv", index=False)
merged_pairs.to_csv("heart_merged_level_homolog_pairs.csv", index=False)

print("Done.")
print(f"Human essential merged lncRNAs: {h_esl.shape[0]}")
print(f"Mouse essential merged lncRNAs: {m_esl.shape[0]}")
print(f"Member-level homologous pairs: {member_pairs.shape[0]}")
print(f"Merged-level homologous lncRNA pairs: {merged_pairs.shape[0]}")
print(f"Ensembl-based matches: {(member_pairs['match_basis'] == 'ensembl').sum()}")
print(f"Gene-name-based matches: {(member_pairs['match_basis'] == 'gene_name').sum()}")




Done.
Human essential merged lncRNAs: 4796
Mouse essential merged lncRNAs: 1554
Member-level homologous pairs: 128
Merged-level homologous lncRNA pairs: 92
Ensembl-based matches: 0
Gene-name-based matches: 128
